In [ ]:
%load_ext autoreload
%autoreload 2

# Import delle librerie e dei moduli locali
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from torch_geometric.loader import DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Individuazione root del progetto per permettere gli import di 'src'
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import config as cfg
from src import utils as ut
from src import data_loader as dl
from src import train as tr
from src import models as mdl

# Setup Riproducibilità e Device
ut.setup_reproducibility(cfg.FIXED_SEED)
device = ut.get_device()
print(f"Addestramento rete neurale attraverso: {device}")

# Caricamento e validazione dataset
dataset = dl.load_and_validate_dataset(str(cfg.DATASET_PATH))

# Analisi features
skewed_features, df_feature_stats = dl.inspect_node_features(dataset)

# Generazione ed esportazione del grafico delle distribuzioni
ut.plot_skewed_features_distributions(
    dataset=dataset, 
    feature_indices=skewed_features, 
    save_path=cfg.RESULTS_DIR / "node_features_distributions.png",
    show = True
)

# Analisi dello sbilanciamento delle classi
distribution = dl.check_class_imbalance(dataset)

# Separazione del Test Set finale (Holdout sicuro)
train_val_dataset, test_dataset = dl.stratified_holdout_split(dataset, cfg.TEST_SIZE, cfg.FIXED_SEED)

In [ ]:
# Griglia di iperparametri (4 combinazioni totali per non appesantire la CPU)
grid_lr = [1e-3, 1e-4]
grid_dropout = [0.1, 0.3]

# Lista di indici delle feature da non normalizzare
excluded_indexes = [4, 29, 30, 31]

results_archive = {}

print("\n=========================================")
print("AVVIO GRID SEARCH: GCN & GAT")
print("=========================================\n")

print("\n[INFO]: Avvio procedura di Cross Validation...")
print("[INFO]: Impostata applicazione Z-Score e Noise Injection in modo isolato all'interno di ciascun fold.\n")

for model_name, model_class in [("GCN", mdl.ProteinGCN), ("GAT", mdl.ProteinGAT)]:
    # print(f"\n Valutazione Architettura: Protein{model_name}")
    print(f"\n VALUTAZIONE ARCHITETTURA: Protein{model_name}")
    for lr in grid_lr:
        for drop in grid_dropout:
            config_name = f"{model_name}_lr_{lr}_drop_{drop}"
            print(f"\n>> Testando configurazione: {config_name}")
            
            f1_res, auc_res = tr.run_cross_validation(
                dataset=train_val_dataset,
                model_class=model_class,
                n_splits=cfg.N_SPLITS,
                batch_size=cfg.DEFAULT_BATCH_SIZE,
                train_fold_fn=tr.train_single_fold,
                seed=cfg.FIXED_SEED,
                lr=lr,
                dropout_p=drop,
                hidden_channels=cfg.DEFAULT_HIDDEN_CHANNELS,
                excluded_features=excluded_indexes,
                noise_level=cfg.NOISE_LEVEL,
                use_log_transform=False,
            )
            results_archive[config_name] = {
                "model": model_name, "lr": lr, "dropout": drop, "channels": cfg.DEFAULT_HIDDEN_CHANNELS,
                "f1_mean": np.mean(f1_res), "f1_std": np.std(f1_res),
                "auc_mean": np.mean(auc_res), "auc_std": np.std(auc_res),
                "f1_scores": f1_res, "auc_scores": auc_res,
            }

print("\n[INFO]: Lavoro completato.")

In [ ]:
df_results = pd.DataFrame.from_dict(results_archive, orient="index").sort_values(by="f1_mean", ascending=False)

# Esportazione della tabella in formato CSV
cfg.RESULTS_DIR.mkdir(exist_ok=True)
df_results.to_csv(cfg.RESULTS_DIR / "grid_search_results.csv", float_format="%.4f")

# Stampa della tabella con le informazioni di interesse
print("\n TABELLA COMPARATIVA DELLE CONFIGURAZIONI (CROSS-VALIDATION) ")
df_display = (
    df_results
    .reset_index()
    .rename(columns={'index': 'Configurazione'})
)

df_styled = (
    df_display.style
    .hide(axis="index")
    .hide(subset=["model", "lr", "dropout", "channels", "f1_scores", "auc_scores"], axis="columns")
    .format(precision=4)
)

display(df_styled)

# Estrazione automatica del modello vincente
selected_config = df_results.iloc[0]
best_model_class = mdl.ProteinGCN if selected_config["model"] == "GCN" else mdl.ProteinGAT

# print(f"selected_config: {selected_config}\n")

print("\n=======================================================================")
print(f" MODELLO VINCITORE SELEZIONATO: {selected_config.name}")
print(f" - Macro F1 Medio in CV: {selected_config['f1_mean']:.4f}")
print(f" - ROC-AUC Medio in CV: {selected_config['auc_mean']:.4f}")
print("=======================================================================\n")

In [ ]:
# Log Trasform + Z-Score Selettiva: Test della trasformazione per mitigare la skewness.

print("=======================================================================")
print("Sperimentazione Log Trasform + Z-Score Selettiva")
print("=======================================================================")

f1_log, auc_log = tr.run_cross_validation(
    dataset=train_val_dataset,
    model_class=best_model_class,
    n_splits=cfg.N_SPLITS,
    batch_size=cfg.DEFAULT_BATCH_SIZE,
    train_fold_fn=tr.train_single_fold,
    seed=cfg.FIXED_SEED,
    lr=selected_config['lr'],
    dropout_p=selected_config['dropout'],
    hidden_channels=cfg.DEFAULT_HIDDEN_CHANNELS,
    excluded_features=excluded_indexes,
    noise_level=cfg.NOISE_LEVEL,
    use_log_transform=True # Attivazione del modulo logaritmico sicuro
)

In [ ]:
print(f" -> Performance CON Log-Transform: Macro F1 Media={np.mean(f1_log):.4f} | ROC-AUC Media={np.mean(auc_log):.4f}")
print(f" -> Performance SENZA Log-Transform: Macro F1 Media={selected_config['f1_mean']:.4f} | ROC-AUC Media={selected_config['auc_mean']:.4f}")

delta_f1 = np.mean(f1_log) - selected_config["f1_mean"]
delta_auc = np.mean(auc_log) - selected_config["auc_mean"]

print(
    f"\nΔ Macro F1 Medio: {delta_f1:+.4f} ({delta_f1 / selected_config['f1_mean'] * 100:+.2f}%)"
)

print(
    f"Δ ROC-AUC Medio: {delta_auc:+.4f} ({delta_auc / selected_config['auc_mean'] * 100:+.2f}%)"
)

print("\n>> Compilazione grafico comparativo...")

baseline_scores=np.array([
    selected_config["f1_mean"],
    selected_config["auc_mean"],
])
comparison_scores=np.array([
    np.mean(f1_log),
    np.mean(auc_log),
])
baseline_std=np.array([
    selected_config["f1_std"],
    selected_config["auc_std"],
])
comparison_std=np.array([
    np.std(f1_log),
    np.std(auc_log),
])
baseline_fold_scores=[
    selected_config["f1_scores"],
    selected_config["auc_scores"],
]
comparison_fold_scores=[
    f1_log,
    auc_log,
]

ut.plot_cv_comparison(
    baseline_scores=baseline_scores,
    comparison_scores=comparison_scores,
    baseline_std=baseline_std,
    comparison_std=comparison_std,
    baseline_fold_scores = baseline_fold_scores,
    comparison_fold_scores= comparison_fold_scores,
    comparison_label="Log Transform",
    title="Baseline vs Log Transform",
    save_path=cfg.RESULTS_DIR / "test_log_transform.png",
    show = True
)

if np.mean(f1_log) > selected_config['f1_mean']:
    final_use_log = True

    current_baseline_f1 = np.mean(f1_log)
    current_baseline_auc = np.mean(auc_log)
    current_baseline_f1_std = np.std(f1_log)
    current_baseline_auc_std = np.std(auc_log)
    current_baseline_f1_scores = f1_log
    current_baseline_auc_scores = auc_log
    current_baseline_label = "Log Transform (Nuova Baseline)"

    print("[+] Log-Transform: ATTIVATA (Ha migliorato le performance)")
else:
    final_use_log = False

    current_baseline_f1 = selected_config['f1_mean']
    current_baseline_auc = selected_config['auc_mean']
    current_baseline_f1_std = selected_config['f1_std']
    current_baseline_auc_std = selected_config['auc_std']
    current_baseline_f1_scores = selected_config['f1_scores']
    current_baseline_auc_scores = selected_config['auc_scores']
    current_baseline_label = "Grid Search Baseline"
    
    print("[-] Log-Transform: DISATTIVATA (Nessun beneficio)")

In [ ]:
# Test di Esclusione delle Feature One-Hot: Verifica dell'impatto della struttura secondaria.

print("\n" + "=======================================================================")
print("Test di Esclusione delle Feature One-Hot")
print("=======================================================================")

# Rimozione colonne 29, 30, 31 (Feature One-Hot)
dataset_without_onehot = dl.drop_features_permanently(train_val_dataset, indexes_to_drop=[29, 30, 31])

f1_no_onehot, auc_no_onehot = tr.run_cross_validation(
    dataset=dataset_without_onehot, # Passiamo il dataset con 29 feature totali
    model_class=best_model_class,
    n_splits=cfg.N_SPLITS,
    batch_size=cfg.DEFAULT_BATCH_SIZE,
    train_fold_fn=tr.train_single_fold,
    seed=cfg.FIXED_SEED,
    lr=selected_config['lr'],
    dropout_p=selected_config['dropout'],
    hidden_channels=cfg.DEFAULT_HIDDEN_CHANNELS,
    excluded_features=[4], # Escludiamo solo la feature categorica rimasta (la ex-04)
    noise_level=cfg.NOISE_LEVEL,
    use_log_transform=final_use_log
)

In [ ]:
print(f"\n -> Modello Base Attuale (32 features): Macro F1 Media={current_baseline_f1:.4f} | ROC-AUC Media={current_baseline_auc:.4f}")
print(f" -> Modello No One-Hot (29 features):   Macro F1 Media = {np.mean(f1_no_onehot):.4f} | ROC-AUC Media={np.mean(auc_no_onehot):.4f}")

delta_f1 = np.mean(f1_no_onehot) - current_baseline_f1
delta_auc = np.mean(auc_no_onehot) - current_baseline_auc

print(f"\nΔ Macro F1 Medio: {delta_f1:+.4f} ({delta_f1 / current_baseline_f1 * 100:+.2f}%)")
print(f"Δ ROC-AUC Medio: {delta_auc:+.4f} ({delta_auc / current_baseline_auc * 100:+.2f}%)")

print("\n>> Compilazione grafico comparativo...")

ut.plot_cv_comparison(
    baseline_scores=np.array([current_baseline_f1, current_baseline_auc]),
    comparison_scores=np.array([np.mean(f1_no_onehot), np.mean(auc_no_onehot)]),
    baseline_std=np.array([current_baseline_f1_std, current_baseline_auc_std]),
    comparison_std=np.array([np.std(f1_no_onehot), np.std(auc_no_onehot)]),
    baseline_fold_scores=[current_baseline_f1_scores, current_baseline_auc_scores],
    comparison_fold_scores=[f1_no_onehot, auc_no_onehot],
    comparison_label="No One-Hot",
    title=f"{current_baseline_label} vs No One-Hot Feature",
    save_path=cfg.RESULTS_DIR / "test_no_onehot.png",
    show=True
)

if np.mean(f1_no_onehot) > current_baseline_f1:
    # Utilizzo dataset "tagliato" e feature 4 unica da escludere dalla Z-Score
    final_dataset = dataset_without_onehot 
    final_excluded_indexes = [4] 

    # Aggiorniamo di nuovo la Baseline
    current_baseline_f1 = np.mean(f1_no_onehot)
    current_baseline_auc = np.mean(auc_no_onehot)
    current_baseline_f1_std = np.std(f1_no_onehot)
    current_baseline_auc_std = np.std(auc_no_onehot)
    current_baseline_f1_scores = f1_no_onehot
    current_baseline_auc_scores = auc_no_onehot
    current_baseline_label = "No One-Hot (Nuova Baseline)"

    print("[+] Esclusione One-Hot: APPLICATA (Le feature erano ridondanti)")
else:
    final_dataset = train_val_dataset
    final_excluded_indexes = [4, 29, 30, 31]
    print("[-] Esclusione One-Hot: NON APPLICATA (Le feature aggiungono segnale utile)")

In [ ]:
# Ulteriore esperimento di configurazione architetturale 

print("\n=======================================================================")
print(f"Avvio analisi di sensibilità architetturale (hidden_channel: {cfg.ALT_HIDDEN_CHANNELS})")
print("=======================================================================")

f1_channels, auc_channels = tr.run_cross_validation(
    dataset=final_dataset,
    model_class=best_model_class,
    n_splits=cfg.N_SPLITS,
    batch_size=cfg.DEFAULT_BATCH_SIZE,
    train_fold_fn=tr.train_single_fold,
    seed=cfg.FIXED_SEED,
    lr=selected_config['lr'],
    dropout_p=selected_config['dropout'],
    hidden_channels=cfg.ALT_HIDDEN_CHANNELS,    # Modifica strutturale
    excluded_features=excluded_indexes,
    noise_level=cfg.NOISE_LEVEL,
    use_log_transform=final_use_log
)

In [ ]:
print(f" -> Performance Modello Base Attuale: Macro F1 Media={current_baseline_f1:.4f} | ROC-AUC Media={current_baseline_auc:.4f}")
print(f" -> Performance con Channels={cfg.ALT_HIDDEN_CHANNELS}:   Macro F1 Media = {np.mean(f1_channels):.4f} | ROC-AUC Media={np.mean(auc_channels):.4f}")

delta_f1 = np.mean(f1_channels) - current_baseline_f1
delta_auc = np.mean(auc_channels) - current_baseline_auc

print(f"\nΔ Macro F1 Medio: {delta_f1:+.4f} ({delta_f1 / current_baseline_f1 * 100:+.2f}%)")
print(f"Δ ROC-AUC Medio: {delta_auc:+.4f} ({delta_auc / current_baseline_auc * 100:+.2f}%)")

print("\n>> Compilazione grafico comparativo...")

ut.plot_cv_comparison(
    baseline_scores=np.array([current_baseline_f1, current_baseline_auc]),
    comparison_scores=np.array([np.mean(f1_channels), np.mean(auc_channels)]),
    baseline_std=np.array([current_baseline_f1_std, current_baseline_auc_std]),
    comparison_std=np.array([np.std(f1_channels), np.std(auc_channels)]),
    baseline_fold_scores=[current_baseline_f1_scores, current_baseline_auc_scores],
    comparison_fold_scores=[f1_channels, auc_channels],
    comparison_label=f"Channels={cfg.ALT_HIDDEN_CHANNELS}",
    title=f"{current_baseline_label} vs Channels={cfg.ALT_HIDDEN_CHANNELS}",
    save_path=cfg.RESULTS_DIR / "test_alt_hidden_channels.png",
    show=True
)

if np.mean(f1_channels) > current_baseline_f1:
    final_channels = cfg.ALT_HIDDEN_CHANNELS
    print(f"[+] STRUTTURA AGGIORNATA: Scelti {cfg.ALT_HIDDEN_CHANNELS} canali per l'addestramento finale.")
else:
    final_channels = cfg.DEFAULT_HIDDEN_CHANNELS
    print(f"[-] STRUTTURA CONFERMATA: Mantenuti i {cfg.DEFAULT_HIDDEN_CHANNELS} canali di default.")

In [ ]:
# Addestramento definitivo con configurazione migliore

print("\n==============================")
print("AVVIO ADDESTRAMENTO FINALE")
print("==============================")

print("\n[INFO]: Configurazione architetturale e pipeline di preprocessing definitiva:")

recap_data = {
    "Modello": selected_config['model'],
    "LR": f"{selected_config['lr']}",
    "Dropout": f"{selected_config['dropout']}", 
    "Hidden_channels": final_channels,
    "Epoche": cfg.DEFAULT_EPOCHS,
    "Noise": f"{cfg.NOISE_LEVEL}",
    "Log-Transform": "Attiva" if final_use_log else "Disattiva",
    "Rimozione One-Hot": "Applicata (29 feat)" if len(final_excluded_indexes) == 1 else "Non Applicata (32 feat)"
}

df_recap = pd.DataFrame([recap_data])
display(df_recap.style.hide(axis="index"))

# Lancio dell'addestramento finale
final_model, history_loss, shifts, final_mean, final_std = tr.train_final_model(    
    dataset=final_dataset,
    model_class=best_model_class,
    lr=selected_config['lr'],
    dropout_p=selected_config['dropout'],
    hidden_channels=final_channels,
    epochs=cfg.DEFAULT_EPOCHS,
    batch_size=cfg.DEFAULT_BATCH_SIZE,
    noise_level=cfg.NOISE_LEVEL,
    weight_decay = 1e-4,
    excluded_features=final_excluded_indexes,
    use_log_transform=final_use_log
)

In [ ]:
# Valutazione sul Test Set

# Rimozione feature One-Hot se attiva
if len(final_excluded_indexes) == 1: 
    test_dataset = dl.drop_features_permanently(test_dataset, indexes_to_drop=[29, 30, 31])

# Applicazione della Log-Transform (usando gli shift del train_val) se attiva
if shifts is not None:
    # Identifichiamo di nuovo le continuous_features per il Test Set
    n_features_test = test_dataset[0].x.shape[1]
    continuous_features = [i for i in range(n_features_test) if i not in final_excluded_indexes]
    test_dataset = dl.apply_safe_log_transform(test_dataset, continuous_features, shifts)

# Preparazione test set 
test_dataset_normalized = dl.apply_z_score(test_dataset, final_mean, final_std, excluded_features=final_excluded_indexes)
test_loader = DataLoader(test_dataset_normalized, batch_size=cfg.DEFAULT_BATCH_SIZE, shuffle=False, drop_last=False)

# Valutazione test set
y_test_true, y_test_pred, y_test_score = tr.evaluate_model(final_model, test_loader)

# Calcolo Metriche Definitive
acc = accuracy_score(y_test_true, y_test_pred)
f1_macro = f1_score(y_test_true, y_test_pred, average="macro")
precision = precision_score(y_test_true, y_test_pred, zero_division=0)
recall = recall_score(y_test_true, y_test_pred)
roc_auc = roc_auc_score(y_test_true, y_test_score)

# Esportazione delle metriche definitive del Test Set 
final_metrics = pd.DataFrame({
    "Accuracy": [acc],
    "Macro_F1": [f1_macro],
    "Precision": [precision],
    "Recall": [recall],
    "ROC_AUC": [roc_auc]
})

final_metrics.to_csv(
    cfg.RESULTS_DIR / "final_test_metrics.csv", index=False, float_format="%.4f"
)

print("=============================================================")
print(" METRICHE DEFINITIVE SUL TEST SET (MODELLO VINCENTE) ")
print("=============================================================")
print(f" - Accuracy  : {acc:.4f}")
print(f" - Macro F1  : {f1_macro:.4f}  <-- Metrica di riferimento")
print(f" - Precision : {precision:.4f}  (Capacità di evitare falsi positivi)")
print(f" - Recall    : {recall:.4f}  (Capacità di scovare tutti gli enzimi)")
print(f" - ROC-AUC   : {roc_auc:.4f}")
print("=============================================================\n")

ut.plot_final_evaluation_metrics(
    history_loss=history_loss,
    y_true=y_test_true,
    y_pred=y_test_pred,
    y_score=y_test_score,
    roc_auc=float(roc_auc),
    save_path=cfg.RESULTS_DIR / "final_evaluation_plots.png",
    show=True
)

print(f"[OK] Tutti gli artefatti d'esame sono stati correttamente esportati nella cartella:\n     {cfg.RESULTS_DIR.resolve()}")